In [0]:
%pip install mlxtend

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import time
import mlflow
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# Carregar a base do catálogo
print("Carregando dados do catálogo...")
df_input = spark.read.table("workspace.default.input_ia_recomendacao").toPandas()

display(df_input.head())

/databricks/python/lib/python3.12/site-packages/pyspark/logger/logger.py:145: DeprecationWarning: The 'warn' method is deprecated, use 'warning' instead
  super().warn(msg, *args, **kwargs)  # type: ignore[arg-type]
{"ts": "2026-09-01 22:30:22.222", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzIzNDE2OTI4MTgyMzQ2MBABIAEyJDAxYTA1ZjE3LWNkN2ItNzQ2OS1hNTg1LWMyMWVlNWI4YjZiNjokYmYzNzEyOWItMTk2Mi0zNTBlLWIxMjUtNTVkYjBhYTQ5MjM0SgwI3qPd1AYQwNHzlwJQAVgBYAFo6c+clOTEow0=.", "context": {}}
{"ts": "2026-09-01 22:30:22.222", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzIzNDE2OTI4MTgyMzQ2MBABIAEyJDAxYTA1ZjE3LWNkN2ItNzQ2OS1hNTg1LWMyMWVlNWI4YjZiNjokYmYzNzEyOWItMTk2Mi0zNTBlLWIxMjUtNTVkYjBhYTQ5MjM0SgwI3qPd1AYQwNHzlwJQAVgBYAFo6c+clOTEow0=.", "context": {}}
{"ts": "2026-09-01 22:30:22.222", "level": "WARNING", "logger": "pyspark.sql.c

Carregando dados do catálogo...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/pandas/types.py:933: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  return pser.astype(pandas_type, copy=False)


,ven_id,pro_id_mestre
0,3727847301010025764,7078
1,3727847301010019125,7078
2,3727847301010019526,7078
3,3727847301010028641,7078
4,3727847301010024252,7078


In [0]:
print("Preparando e filtrando a matriz de transações...")

# 1. Filtra vendas com ao menos 2 itens
contagem_itens = df_input.groupby('ven_id').size()
vendas_validas = contagem_itens[contagem_itens >= 2].index
df_treino = df_input[df_input['ven_id'].isin(vendas_validas)].copy()

# Apenas para você conferir a mágica acontecendo no log:
qtd_ignoradas = df_input['ven_id'].nunique() - len(vendas_validas)
print(f"-> Vendas de item único removidas: {qtd_ignoradas}")
print(f"-> Vendas válidas para associação: {len(vendas_validas)}")

# 2. Pivotando a tabela para a matriz cruzada (agora usando df_treino!)
basket = (df_treino.groupby(['ven_id', 'pro_id_mestre'])['pro_id_mestre']
          .count().unstack().reset_index().fillna(0)
          .set_index('ven_id'))

# 3. Transforma em booleanos
basket_sets = basket > 0

print(f"-> Matriz Pronta: {basket_sets.shape[0]} transações válidas e {basket_sets.shape[1]} produtos.")

/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWar

Preparando e filtrando a matriz de transações...
-> Vendas de item único removidas: 17662
-> Vendas válidas para associação: 13587
-> Matriz Pronta: 13587 transações válidas e 1811 produtos.


In [0]:
# 1. Verificando o tamanho da base carregada do catálogo
print("--- DIAGNÓSTICO DA BASE DO CATÁLOGO ---")
print(f"Total de registros: {len(df_input)}")
print(f"Vendas únicas (ven_id): {df_input['ven_id'].nunique()}")
print(f"Produtos únicos (pro_id_mestre): {df_input['pro_id_mestre'].nunique()}\n")

# 2. Verificando a Matriz de Treinamento
print("--- DIAGNÓSTICO DA MATRIZ ---")
print(f"Linhas da Matriz (Transações válidas): {basket_sets.shape[0]}")
print(f"Colunas da Matriz (Produtos válidos): {basket_sets.shape[1]}")
print(f"Total de Trues na Matriz: {basket_sets.values.sum()}")

# 3. Teste Rápido Forçando um Parâmetro Diferente
print("\n--- TESTE FORÇADO DE MUDANÇA DE PARÂMETRO ---")
freq_items_teste = fpgrowth(basket_sets, min_support=0.01, use_colnames=True) # Usando 1% forçado
regras_teste = association_rules(freq_items_teste, metric="lift", min_threshold=1.0) if not freq_items_teste.empty else []
print(f"Regras geradas com suporte de 0.01: {len(regras_teste)}")

--- DIAGNÓSTICO DA BASE DO CATÁLOGO ---
Total de registros: 59528
Vendas únicas (ven_id): 31249
Produtos únicos (pro_id_mestre): 1939

--- DIAGNÓSTICO DA MATRIZ ---
Linhas da Matriz (Transações válidas): 13587
Colunas da Matriz (Produtos válidos): 1811
Total de Trues na Matriz: 41866

--- TESTE FORÇADO DE MUDANÇA DE PARÂMETRO ---
Regras geradas com suporte de 0.01: 20


/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)


In [0]:
import time
import tracemalloc
import mlflow
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# =========================================================
# PARÂMETROS ORIGINAIS
# =========================================================
MIN_SUPPORT = 0.001
MIN_THRESHOLD = 1.3
METRIC = "lift"

experiment_name = "/Users/michelcirilo@hotmail.com/Comparativo_Apriori_vs_FPGrowth"
mlflow.set_experiment(experiment_name)

# --- Contador Incremental Dinâmico (Memória do Notebook) ---
try:
    contador_iteracao += 1
except NameError:
    contador_iteracao = 1

iteracao = contador_iteracao
print(f"Iniciando Iteração R{iteracao} do Experimento...\n")

# ==========================================
# RUN 1: APRIORI
# ==========================================
print("Rodando Apriori...")
nome_run_ap = f"Apriori_MS_{MIN_SUPPORT}_MT_{MIN_THRESHOLD}_R{iteracao}"

with mlflow.start_run(run_name=nome_run_ap):
    mlflow.log_param("algoritmo", "Apriori")
    mlflow.log_param("min_support", MIN_SUPPORT)
    mlflow.log_param("min_threshold", MIN_THRESHOLD)
    mlflow.log_param("metric", METRIC)
    
    tracemalloc.start()
    start_time = time.time()
    
    freq_items_ap = apriori(basket_sets, min_support=MIN_SUPPORT, use_colnames=True)
    rules_ap = association_rules(freq_items_ap, metric=METRIC, min_threshold=MIN_THRESHOLD) if not freq_items_ap.empty else pd.DataFrame()
    
    # Cálculos arredondados (2 casas decimais)
    tempo_apriori = round(time.time() - start_time, 2)
    _, pico_memoria_apriori = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    pico_memoria_apriori_mb = round(pico_memoria_apriori / (1024 * 1024), 2)
    
    # Salva no MLflow
    mlflow.log_metric("num_frequent_itemsets", len(freq_items_ap))
    mlflow.log_metric("num_regras", len(rules_ap))
    mlflow.log_metric("tempo_execucao_segundos", tempo_apriori)
    mlflow.log_metric("pico_memoria_mb", pico_memoria_apriori_mb)
    
    print(f"✅ Apriori | Tempo: {tempo_apriori}s | Pico Memória: {pico_memoria_apriori_mb} MB | Regras: {len(rules_ap)}")


# ==========================================
# RUN 2: FP-GROWTH
# ==========================================
print("\nRodando FP-Growth...")
nome_run_fp = f"FP-Growth_MS_{MIN_SUPPORT}_MT_{MIN_THRESHOLD}_R{iteracao}"

with mlflow.start_run(run_name=nome_run_fp):
    mlflow.log_param("algoritmo", "FP-Growth")
    mlflow.log_param("min_support", MIN_SUPPORT)
    mlflow.log_param("min_threshold", MIN_THRESHOLD)
    mlflow.log_param("metric", METRIC)
    
    tracemalloc.start()
    start_time = time.time()
    
    freq_items_fp = fpgrowth(basket_sets, min_support=MIN_SUPPORT, use_colnames=True)
    rules_fp = association_rules(freq_items_fp, metric=METRIC, min_threshold=MIN_THRESHOLD) if not freq_items_fp.empty else pd.DataFrame()
    
    # Cálculos arredondados (2 casas decimais)
    tempo_fpgrowth = round(time.time() - start_time, 2)
    _, pico_memoria_fpgrowth = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    pico_memoria_fpgrowth_mb = round(pico_memoria_fpgrowth / (1024 * 1024), 2)
    
    # Salva no MLflow
    mlflow.log_metric("num_frequent_itemsets", len(freq_items_fp))
    mlflow.log_metric("num_regras", len(rules_fp))
    mlflow.log_metric("tempo_execucao_segundos", tempo_fpgrowth)
    mlflow.log_metric("pico_memoria_mb", pico_memoria_fpgrowth_mb)
    
    print(f"✅ FP-Growth | Tempo: {tempo_fpgrowth}s | Pico Memória: {pico_memoria_fpgrowth_mb} MB | Regras: {len(rules_fp)}")


# ==========================================
# COMPARAÇÃO FINAL
# ==========================================
print("\n" + "="*50)
print(f"🏆 COMPARAÇÃO - ITERAÇÃO R{iteracao}")
print("="*50)
print(f"{'Métrica':<15} | {'Apriori':<12} | {'FP-Growth':<12}")
print("-" * 45)
print(f"{'Tempo (s)':<15} | {tempo_apriori:<12} | {tempo_fpgrowth:<12}")
print(f"{'Memória (MB)':<15} | {pico_memoria_apriori_mb:<12} | {pico_memoria_fpgrowth_mb:<12}")
print("-" * 45)

# Calculando multiplicadores arredondados como números inteiros
if tempo_fpgrowth < tempo_apriori:
    vel_mult = round(tempo_apriori / tempo_fpgrowth) if tempo_fpgrowth > 0 else 0
    print(f"🚀 VELOCIDADE : FP-Growth foi {vel_mult}x mais RÁPIDO!")
elif tempo_apriori < tempo_fpgrowth:
    vel_mult = round(tempo_fpgrowth / tempo_apriori) if tempo_apriori > 0 else 0
    print(f"🚀 VELOCIDADE : Apriori foi {vel_mult}x mais RÁPIDO!")
else:
    print("🚀 VELOCIDADE : Empate técnico!")

mem_fp = max(pico_memoria_fpgrowth_mb, 0.01) # Evita erro de divisão por zero
mem_ap = max(pico_memoria_apriori_mb, 0.01)

if pico_memoria_fpgrowth_mb < pico_memoria_apriori_mb:
    mem_mult = round(mem_ap / mem_fp)
    print(f"🧠 MEMÓRIA    : FP-Growth consumiu {mem_mult}x MENOS memória!")
elif pico_memoria_apriori_mb < pico_memoria_fpgrowth_mb:
    mem_mult = round(mem_fp / mem_ap)
    print(f"🧠 MEMÓRIA    : Apriori consumiu {mem_mult}x MENOS memória!")
else:
    print("🧠 MEMÓRIA    : Empate técnico no consumo!")

/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  ret = getattr(ast, func_name)(*args, **kwargs)
/databricks/python/lib/python3.12/site-packages/pyccolo/_fast/fast_ast.py:59: DeprecationWar

Iniciando Iteração R1 do Experimento...

Rodando Apriori...
✅ Apriori | Tempo: 142.45s | Pico Memória: 7000.29 MB | Regras: 2050

Rodando FP-Growth...
✅ FP-Growth | Tempo: 1.28s | Pico Memória: 6.03 MB | Regras: 2050

🏆 COMPARAÇÃO - ITERAÇÃO R1
Métrica         | Apriori      | FP-Growth   
---------------------------------------------
Tempo (s)       | 142.45       | 1.28        
Memória (MB)    | 7000.29      | 6.03        
---------------------------------------------
🚀 VELOCIDADE : FP-Growth foi 111x mais RÁPIDO!
🧠 MEMÓRIA    : FP-Growth consumiu 1161x MENOS memória!
